# 🧾 Resume / Candidate Screening System (Task-03)
## Automatically Screen & Rank Resumes Based on a Given Job Role

### ✅ Objective
Build an ML/NLP system that can:
- Parse and clean resume text
- Extract important skills from resumes
- Match resumes with a given job description
- Rank candidates based on role fit score
- Identify missing (skill gap) for each candidate

### ✅ Tools
- Python
- Pandas, NumPy
- Scikit-learn (TF-IDF + Cosine Similarity)
- Jupyter Notebook

# 🔧 Step 1: Import Required Libraries

In [1]:
import re
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 📥 Step 2: Load Resume Dataset (CSV)
We will load the dataset from:
`resume_screening_automation.csv`

In [2]:
DATA_PATH = "resume_screening_automation.csv"

df = pd.read_csv(DATA_PATH)
print("✅ Dataset Shape:", df.shape)
df.head()

✅ Dataset Shape: (100, 3)


,resume_text,job_title,suitability_score
0,"Experienced in machine learning, data analysis...",Data Scientist,0.37
1,"Experienced in roadmap, agile, stakeholders.",Product Manager,0.11
2,"Experienced in stakeholders, roadmap, market r...",Product Manager,0.32
3,"Experienced in agile, MVP, stakeholders.",Product Manager,0.90
4,"Experienced in python, pandas, statistics.",Data Scientist,0.11


# 🧾 Step 3: Understand Dataset Columns
We need to identify:
- Resume Text column (resume content)
- Optional job role / category column (if available)
- Optional skills column (if available)

In [3]:
df.columns

Index(['resume_text', 'job_title', 'suitability_score'], dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   resume_text        100 non-null    str    
 1   job_title          100 non-null    str    
 2   suitability_score  100 non-null    float64
dtypes: float64(1), str(2)
memory usage: 2.5 KB


In [5]:
df.isna().sum().sort_values(ascending=False).head(20)

resume_text          0
job_title            0
suitability_score    0
dtype: int64

# 🧠 Step 4: Select Resume Text Column Automatically
This cell tries to detect the best resume text column.
If it picks wrong, manually set:

RESUME_COL = "your_resume_column"

In [6]:
possible_resume_cols = [c for c in df.columns if any(k in c.lower() for k in ["resume", "cv", "text", "content", "profile", "description"])]
print("Possible resume columns:", possible_resume_cols)

RESUME_COL = possible_resume_cols[0] if len(possible_resume_cols) else df.columns[0]
print("✅ Selected RESUME_COL:", RESUME_COL)

df[[RESUME_COL]].head()

Possible resume columns: ['resume_text']
✅ Selected RESUME_COL: resume_text


,resume_text
0,"Experienced in machine learning, data analysis..."
1,"Experienced in roadmap, agile, stakeholders."
2,"Experienced in stakeholders, roadmap, market r..."
3,"Experienced in agile, MVP, stakeholders."
4,"Experienced in python, pandas, statistics."


# 🧹 Step 5: Clean and Preprocess Resume Text
We will:
- lowercase
- remove emails, links
- remove special characters
- remove extra spaces

In [7]:
def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\S+@\S+", " ", s)              # remove emails
    s = re.sub(r"http\S+|www\S+", " ", s)       # remove URLs
    s = re.sub(r"[^a-z0-9\s]", " ", s)          # remove special chars
    s = re.sub(r"\s+", " ", s).strip()          # normalize spaces
    return s

df["clean_resume"] = df[RESUME_COL].apply(clean_text)
df[["clean_resume"]].head()

,clean_resume
0,experienced in machine learning data analysis ...
1,experienced in roadmap agile stakeholders
2,experienced in stakeholders roadmap market res...
3,experienced in agile mvp stakeholders
4,experienced in python pandas statistics


# 🎯 Step 6: Define the Target Job Role / Job Description
We will type a job description and match resumes against it.

✅ You can change this for different job roles.

In [8]:
job_role = "Data Scientist"

job_description = """
We are looking for a Data Scientist with strong skills in:
Python, Machine Learning, Pandas, NumPy, Statistics,
SQL, Data Visualization, Model Evaluation, Feature Engineering,
and basic knowledge of Deep Learning.
"""

job_description_clean = clean_text(job_description)
print("✅ Job Role:", job_role)
print("✅ Job Description:", job_description_clean[:300], "...")

✅ Job Role: Data Scientist
✅ Job Description: we are looking for a data scientist with strong skills in python machine learning pandas numpy statistics sql data visualization model evaluation feature engineering and basic knowledge of deep learning ...


# 🤖 Step 7: Convert Text to TF-IDF and Rank Resumes
We will:
- Convert job description and resumes to TF-IDF vectors
- Compute cosine similarity
- Rank candidates by similarity score

In [9]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=50000)

resume_vectors = vectorizer.fit_transform(df["clean_resume"])
job_vector = vectorizer.transform([job_description_clean])

scores = cosine_similarity(job_vector, resume_vectors).flatten()

df["match_score"] = scores
df_sorted = df.sort_values("match_score", ascending=False).reset_index(drop=True)

df_sorted[["match_score", RESUME_COL]].head(10)

,match_score,resume_text
0,0.773028,"Experienced in python, machine learning, pandas."
1,0.563724,"Experienced in python, machine learning, stati..."
2,0.563724,"Experienced in python, machine learning, stati..."
3,0.563160,"Experienced in pandas, python, machine learning."
4,0.463154,"Experienced in python, data analysis, machine ..."
5,0.463154,"Experienced in python, data analysis, machine ..."
6,0.456894,"Experienced in pandas, machine learning, data ..."
7,0.456894,"Experienced in pandas, data analysis, machine ..."
8,0.456894,"Experienced in pandas, machine learning, data ..."
9,0.450006,"Experienced in machine learning, python, data ..."


# 🏆 Step 8: Display Top Ranked Candidates
This is the final screening output (Top N candidates).

In [10]:
TOP_N = 10
top_candidates = df_sorted.head(TOP_N).copy()

top_candidates[["match_score"]].head(TOP_N)

,match_score
0,0.773028
1,0.563724
2,0.563724
3,0.563160
4,0.463154
5,0.463154
6,0.456894
7,0.456894
8,0.456894
9,0.450006


In [11]:
for i in range(TOP_N):
    print(f"\n✅ Rank {i+1} | Score: {top_candidates.loc[i,'match_score']:.4f}")
    print(str(top_candidates.loc[i, RESUME_COL])[:500], "...")


✅ Rank 1 | Score: 0.7730
Experienced in python, machine learning, pandas. ...

✅ Rank 2 | Score: 0.5637
Experienced in python, machine learning, statistics. ...

✅ Rank 3 | Score: 0.5637
Experienced in python, machine learning, statistics. ...

✅ Rank 4 | Score: 0.5632
Experienced in pandas, python, machine learning. ...

✅ Rank 5 | Score: 0.4632
Experienced in python, data analysis, machine learning. ...

✅ Rank 6 | Score: 0.4632
Experienced in python, data analysis, machine learning. ...

✅ Rank 7 | Score: 0.4569
Experienced in pandas, machine learning, data analysis. ...

✅ Rank 8 | Score: 0.4569
Experienced in pandas, data analysis, machine learning. ...

✅ Rank 9 | Score: 0.4569
Experienced in pandas, machine learning, data analysis. ...

✅ Rank 10 | Score: 0.4500
Experienced in machine learning, python, data analysis. ...


# 🧠 Step 9: Skill Extraction and Skill Gap Identification
We will:
- Maintain a skill list
- Extract skills present in each resume
- Compare with required skills from job description
- Show missing skills (skill gap)

In [12]:
SKILLS = [
    "python", "machine learning", "deep learning", "sql", "pandas", "numpy",
    "statistics", "data visualization", "matplotlib", "seaborn",
    "scikit learn", "sklearn", "feature engineering", "model evaluation",
    "nlp", "tensorflow", "pytorch", "power bi", "excel"
]

required_skills = [s for s in SKILLS if s in job_description_clean]
print("✅ Required Skills (from job description):", required_skills)

✅ Required Skills (from job description): ['python', 'machine learning', 'deep learning', 'sql', 'pandas', 'numpy', 'statistics', 'data visualization', 'feature engineering', 'model evaluation']


In [13]:
def extract_skills(text: str, skills_list):
    found = []
    t = text.lower()
    for skill in skills_list:
        if skill in t:
            found.append(skill)
    return sorted(set(found))

top_candidates["skills_found"] = top_candidates["clean_resume"].apply(lambda x: extract_skills(x, SKILLS))
top_candidates["missing_skills"] = top_candidates["skills_found"].apply(lambda x: sorted(set(required_skills) - set(x)))

top_candidates[["match_score", "skills_found", "missing_skills"]].head(10)

,match_score,skills_found,missing_skills
0,0.773028,"[machine learning, pandas, python]","[data visualization, deep learning, feature en..."
1,0.563724,"[machine learning, python, statistics]","[data visualization, deep learning, feature en..."
2,0.563724,"[machine learning, python, statistics]","[data visualization, deep learning, feature en..."
3,0.563160,"[machine learning, pandas, python]","[data visualization, deep learning, feature en..."
4,0.463154,"[machine learning, python]","[data visualization, deep learning, feature en..."
5,0.463154,"[machine learning, python]","[data visualization, deep learning, feature en..."
6,0.456894,"[machine learning, pandas]","[data visualization, deep learning, feature en..."
7,0.456894,"[machine learning, pandas]","[data visualization, deep learning, feature en..."
8,0.456894,"[machine learning, pandas]","[data visualization, deep learning, feature en..."
9,0.450006,"[machine learning, python]","[data visualization, deep learning, feature en..."


# 📌 Step 10: Final Human-Friendly Output (Ranking + Skills + Skill Gap)

In [14]:
for i in range(TOP_N):
    print("="*90)
    print(f"🏅 Rank {i+1} | Match Score: {top_candidates.loc[i,'match_score']:.4f}")
    print("\n✅ Skills Found:")
    print(top_candidates.loc[i, "skills_found"])
    print("\n❌ Missing Skills (Skill Gap):")
    print(top_candidates.loc[i, "missing_skills"])
    print("\n📄 Resume Preview:")
    print(str(top_candidates.loc[i, RESUME_COL])[:500], "...")

🏅 Rank 1 | Match Score: 0.7730

✅ Skills Found:
['machine learning', 'pandas', 'python']

❌ Missing Skills (Skill Gap):
['data visualization', 'deep learning', 'feature engineering', 'model evaluation', 'numpy', 'sql', 'statistics']

📄 Resume Preview:
Experienced in python, machine learning, pandas. ...
🏅 Rank 2 | Match Score: 0.5637

✅ Skills Found:
['machine learning', 'python', 'statistics']

❌ Missing Skills (Skill Gap):
['data visualization', 'deep learning', 'feature engineering', 'model evaluation', 'numpy', 'pandas', 'sql']

📄 Resume Preview:
Experienced in python, machine learning, statistics. ...
🏅 Rank 3 | Match Score: 0.5637

✅ Skills Found:
['machine learning', 'python', 'statistics']

❌ Missing Skills (Skill Gap):
['data visualization', 'deep learning', 'feature engineering', 'model evaluation', 'numpy', 'pandas', 'sql']

📄 Resume Preview:
Experienced in python, machine learning, statistics. ...
🏅 Rank 4 | Match Score: 0.5632

✅ Skills Found:
['machine learning', 'pandas'

# 💾 Step 11: Save Ranked Candidates to CSV
This helps in real HR workflow:
We export the ranking results for top candidates.

In [15]:
output_cols = ["match_score", RESUME_COL, "skills_found", "missing_skills"]
top_candidates[output_cols].to_csv("ranked_candidates.csv", index=False)

print("✅ Saved file: ranked_candidates.csv")

✅ Saved file: ranked_candidates.csv


# ✅ Final Summary

### This system can:
✅ Clean and preprocess resume text  
✅ Accept a job role and job description  
✅ Rank candidates using TF-IDF + Cosine Similarity  
✅ Extract skills from resumes  
✅ Identify skill gaps (missing required skills)  
✅ Export final ranked candidates to CSV  

### Next Improvements (Optional)
- Use spaCy for better skill extraction (NER)
- Use BERT embeddings for better matching
- Create a Streamlit web app for recruiters